# Multiple inheritance and MRO `[ext.]`

A class can have several parents. Most often: small classes (mixins) that add
one ability each.

In [1]:
class ToDictMixin:
    def to_dict(self):
        result = {}
        for name, value in vars(self).items():
            if not name.startswith("_"):
                result[name] = value
        return result


class ReprMixin:
    def __repr__(self):
        parts = []
        for name, value in vars(self).items():
            parts.append(f"{name}={value!r}")
        return f"{type(self).__name__}({', '.join(parts)})"


class Ticket(ToDictMixin, ReprMixin):
    def __init__(self, category, price):
        self.category = category
        self.price = price


ticket = Ticket("full", 22.0)
print(ticket)
print(ticket.to_dict())

Ticket(category='full', price=22.0)
{'category': 'full', 'price': 22.0}


## Which method wins?

In [2]:
class Announcer:
    def describe(self):
        return "announcer"


class Performer:
    def describe(self):
        return "performer"


class Host(Announcer, Performer):
    pass


print(Host().describe())
for klass in Host.__mro__:
    print(klass.__name__)

announcer
Host
Announcer
Performer
object


The **MRO** (method resolution order): the class, then its parents from left to right, `object` last.

In [3]:
class Base:
    def hello(self):
        return ["Base"]


class Left(Base):
    def hello(self):
        return ["Left"] + super().hello()


class Right(Base):
    def hello(self):
        return ["Right"] + super().hello()


class Both(Left, Right):
    def hello(self):
        return ["Both"] + super().hello()


print(Both().hello())

['Both', 'Left', 'Right', 'Base']


`super()` in `Left` calls `Right`, not `Base`: it follows the MRO of the
object. Each class is visited once. For the project, simple inheritance is
enough.